## Factory Method

(The Mapping Function)

---

Let $K$ be a set of input keys and $\mathcal{T}$ be a set of possible object types. The Factory Method defines a mapping:

$$f : K \rightarrow \mathcal{T}$$

$$f(k) = T_k() \quad \text{where} \quad T_k \in \mathcal{T}$$

$$f(\text{"dog"}) = \text{Dog()}, \quad f(\text{"cat"}) = \text{Cat()}, \quad f(\text{"bird"}) = \text{Bird()}$$

**Total function condition** — every valid key maps to exactly one type:

$$\forall k \in K, \quad \exists! \, T_k \in \mathcal{T} \quad \text{such that} \quad f(k) = T_k()$$

**Fresh object per call** — unlike Singleton, every call produces a new object:

$$\text{addr}(f(k)_1) \neq \text{addr}(f(k)_2)$$

**Isolation condition** — the caller only knows $K$ (the keys). The factory owns the knowledge of $\mathcal{T}$ (the types):

$$\text{caller} \xrightarrow{k} f \xrightarrow{T_k} \text{object}$$

This decoupling is the core value of the pattern.


### Exercise 1 — Animal Factory
---

**Scenario:** You're building a virtual pet app. Depending on the user's choice, you need to create a different animal. You don't want the rest of your code worrying about *which* animal class to use.

**Your task:** Create `Dog`, `Cat`, and `Bird` classes, each with a `speak()` method. Write a factory that returns the right one based on a string.

```python
animal = AnimalFactory.create("dog")
animal.speak()  # Woof!

animal = AnimalFactory.create("cat")
animal.speak()  # Meow!
```

In [ ]:
# Define class for each animal with a speak method

class Dog:
    def speak(self):
        return "Woof!"

class Cat:
    def speak(self):
        return "Meow!"

class Bird:
    def speak(self):
        return "Tweet!"

# Now we will create a dictionary of the animals and put them in a factory that will return the right animal based on the key
class Factory:
    # Define a dictionary of the animals and put them in a factory that will return the right animal based on the key
    T = {
        "1": Dog,
        "2": Cat,
        "3": Bird
    }    

    
    @staticmethod
    def create(k):
        if k not in Factory.T:
            raise ValueError(f"The animal type: {k} is not supported")
        return Factory.T[k]()

print(Factory.create("1").speak())
print(Factory.create("2").speak())
print(Factory.create("3").speak())

# Testing the factory to see if it is a singleton
a= Factory.create("1")
b= Factory.create("1")
print(id(a))
print(id(b))
print(a is b)

Woof!
Meow!
Tweet!
4437234448
4437234768
False


### Exercise 2 — Payment Factory
---
**Scenario:** You're building a checkout system. Users can pay by Credit Card, PayPal, or Crypto. Each has a different `pay(amount)` method, but the rest of the app shouldn't care which one is being used.

**Your task:** Build `CreditCard`, `PayPal`, and `Crypto` classes, each with a `pay(amount)` method. Write a factory that returns the right one.

```python
processor = PaymentFactory.create("paypal")
processor.pay(99.99)  # Paying $99.99 via PayPal
```

In [28]:
# Define class for each payment method with a pay method

class CreditCard:
    def pay(self, amount):
        return f"Paying ${amount} via Credit Card"

class PayPal:
    def pay(self, amount):
        return f"Paying ${amount} via PayPal"

class Crypto:
    def pay(self, amount):
        return f"Paying ${amount} via Crypto"


# Define a factory that will return the right payment method based on the key
class PaymentFactory:
    # Define a dictionary of the payment methods and put them in a factory that will return the right payment method based on the key
    
    T = {
    "1": CreditCard,
    "2": PayPal,
    "3": Crypto
}

    @staticmethod
    def create(k):
        if k not in PaymentFactory.T:
            raise ValueError(f"The payment method: {k} is not supported") 
        return PaymentFactory.T[k]()

amount = 99.99
print(PaymentFactory.create("1").pay(amount))
print(PaymentFactory.create("2").pay(amount))
print(PaymentFactory.create("3").pay(amount))

Paying $99.99 via Credit Card
Paying $99.99 via PayPal
Paying $99.99 via Crypto


---
- Think about edge cases — should `"bitcoin"` map to `Crypto`? Handle them explicitly.
- Try adding a base class `PaymentProcessor` with an abstract `pay()` method using Python's `abc` module.

In [34]:
from pydantic import BaseModel
from typing import Literal
from abc import ABC, abstractmethod

#--------------------------------

class Money(BaseModel):
    value: float
    currency: Literal['AUD', 'Bitcoin']

class PaymentProcessor:
    @abstractmethod
    def pay(self, amount: Money):
        return f"Paying ${amount.value} {amount.currency} via {self.__class__.__name__}"

#--------------------------------

class CreditCard(PaymentProcessor):
    def pay(self, amount: Money):
        return super().pay(amount)

class PayPal(PaymentProcessor):
    def pay(self, amount: Money):
        return super().pay(amount)

class Crypto(PaymentProcessor):
    def pay(self, amount: Money):
        return super().pay(amount)

#--------------------------------
class PaymentFactory:

    T = {
    "1": CreditCard,
    "2": PayPal,
    "3": Crypto
}

    @staticmethod
    def create(k, amount: Money):
        if k not in PaymentFactory.T:
            raise ValueError(f"The payment method: {k} is not supported") 

        if amount.currency == 'Bitcoin':
            return PaymentFactory.T['3']().pay(amount)
        else:
            return PaymentFactory.T[k]().pay(amount)

#--------------------------------
amount_1 = Money(value=99.99, currency='Bitcoin')
print(PaymentFactory.create("1", amount_1))
print(PaymentFactory.create("2", amount_1))
print(PaymentFactory.create("3", amount_1))

amount_2 = Money(value=99.99, currency='AUD')
print(PaymentFactory.create("1", amount_2))
print(PaymentFactory.create("2", amount_2))
print(PaymentFactory.create("3", amount_2))


Paying $99.99 Bitcoin via Crypto
Paying $99.99 Bitcoin via Crypto
Paying $99.99 Bitcoin via Crypto
Paying $99.99 AUD via CreditCard
Paying $99.99 AUD via PayPal
Paying $99.99 AUD via Crypto
